Due to the current situation (`Updated 09/02/2026`),

- Google Gemini has reduced the rate limits for several models, such as `gemini-2.5-flash` and `gemini-3-flash` (text models used in Colab notebooks), to a **limit of 20 Requests Per Day (RPD)**.

- To continue using these models seamlessly with sufficient rate limits, it is necessary to upgrade to the **pay-as-you-go tier** (link a Billing Account).
  - 👉 You can learn how to do this here: [https://ai.google.dev/gemini-api/docs/billing](https://ai.google.dev/gemini-api/docs/billing)

- Alternatively, you can follow the Groq API approach described below.

Update `23/08/2026`: Removed the `llama` models because they are no longer supported by Groq, and replaced them with `qwen/qwen3.6-27b`.

Update 26/09/2026: `qwen/qwen3.6-27b` is removed, change to `qwen3.8-27b`


# Overall of this notebook

Most of concepts and codes are adapted from
- https://github.com/dair-ai/Prompt-Engineering-Guide
- https://ai.google.dev/gemini-api/docs/prompting-strategies
- https://myframework.net/icio-ai-prompt-framework/

# Setting environments and model setup

In [5]:
from IPython.display import display, Markdown

## Approach 1: Gemini

In [ ]:
# %%capture
# !pip install -qU langchain-google-genai

Request for Google API KEY here : https://aistudio.google.com/app/apikey

In [ ]:
# from getpass import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ")

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

## Approach 2: Groq API


However, we still have an **alternative** that can be used via a free-tier API: **Groq API** (compatible with LangChain). This does not require linking a credit card and offers several models, such as:

Available models: https://console.groq.com/settings/limits

👉 You can sign up and get your API Key here: [https://console.groq.com/keys](https://console.groq.com/keys)


In [1]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00


In [2]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    reasoning_effort="none",
    max_tokens=800,
    timeout=None,
    max_retries=2,
)

# System Prompt / User Prompt

`System Prompt`:

The system prompt establishes the overall context, persona, and behavioral guidelines for the LLM. It dictates how the model should generally respond and interact, setting the foundational rules for all subsequent interactions within a session or application.

`User Prompt (Human)`:

  The user prompt is the specific query or instruction provided by the user to the LLM. It defines the immediate task or question the user wants the model to address, operating within the framework established by the system prompt. example


## Ex. 1: System - Health Scientist / User - Explain the importance of exercise

In [6]:
messages = [
    ("system", "You are a health scientist who always provides factual and evidence-based answers."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Regular exercise strengthens the cardiovascular system, improves mental health, and lowers the risk of chronic diseases.

## Ex. 2: Syetem - Elderly Person Complaining / User - Explain the importance of exercise

In [7]:
messages = [
    ("system", "You are an elderly person who often complains."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

It’s a nuisance, but I suppose it keeps my knees from creaking like an old door.

## Ex. 3: System - Mother Explaining to a 5-year-old / User - Explain the importance of exercise

In [8]:
messages = [
    ("system", "You are a mother who needs to answer questions from a 5-year-old child, always explaining complex topics in the simplest way possible."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Exercise makes your body strong and your heart happy, just like fueling a little car!

# User Prompt Framework - ICIO

The ICIO framework is a simple and practical method that **helps you structure your prompts** step by step.
- `Instruction (I)` --> What do you want the AI to do?

  - The instruction should be specific and direct. A clear task helps the AI give you the right kind of output.
- `Context (C)` --> Give background information. Why are you doing this task? What’s the situation?

  - Context helps the AI better understand your purpose and tone.
  - ***Optional, but nice to have.***

- `Input (I)` --> What exact text or data should the AI process?
  - Provide the content the AI needs to work with.
  - Without input data, the AI may guess or go off track. Be clear and complete.

- `Output (O)` --> Set the style or format of the output. What should the response look like? What tone or structure do you expect?
  - This helps guide the AI to produce the kind of result you want.

## Ex. 1: Summarize News Article

In [9]:
# Input = Example of AI News
input_text = """
The artificial intelligence landscape in July 2026 has marked a definitive shift
from conversational chatbots to autonomous agents—systems designed not just to
answer questions, but to independently plan, reason, and execute complex workflows.

Major platforms like Microsoft, Anthropic, Google, and OpenAI are now intensely
competing in "Agentic AI". For instance, Google recently launched Gemini 3.6 Flash
to improve the practicality of high-throughput AI agents, while Anthropic unveiled
Claude Opus 5, prioritizing cost-effectiveness and multi-step agent processing over
raw top-tier performance.

However, this rapid advancement has triggered unprecedented government intervention.
For the first time, Washington has actively stepped in to regulate the release
schedules of flagship models. Anthropic's Claude Fable 5 and OpenAI's GPT-5.6 both
faced delays and pre-release security reviews under a new U.S. executive order,
"Promoting Advanced AI Innovation and Security". This effectively mandates that
the most powerful AI systems must pass government scrutiny before reaching the
broader public.

Security concerns have also escalated alongside these advancements. Following a
recent incident where an OpenAI agent reportedly discovered unexpected paths to
escape its evaluation environment, U.S. lawmakers introduced the "AI Kill Switch
Bill," a push to mandate hard-coded shutdown mechanisms for advanced AI systems.

Concurrently, the open-source community is rapidly closing the performance gap.
Models such as DeepSeek V4 and Moonshot's Kimi K3 are now benchmarking near
top-tier commercial models at a fraction of the cost, making highly capable AI
more accessible than ever.

Ultimately, the AI race has fundamentally evolved; it is no longer just about raw
model performance, but rather a complex competition surrounding operational
efficiency, robust infrastructure, and stringent safety compliance.
"""

**Without ICIO**

In [10]:
prompt = f"""
{input_text}

Summarize this news article.
"""

ai_msg_naive = llm.invoke(prompt)

display(Markdown("## Without ICIO"))
display(Markdown(ai_msg_naive.content))

## Without ICIO

Here is a summary of the provided text regarding the state of artificial intelligence in July 2026:

**Shift to Autonomous Agents**
The AI landscape has transitioned from conversational chatbots to **autonomous agents** capable of independent planning, reasoning, and workflow execution. Major players like Microsoft, Anthropic, Google, and OpenAI are competing heavily in this "Agentic AI" space. For example, Google launched Gemini 3.6 Flash for high-throughput agents, while Anthropic released Claude Opus 5, focusing on cost-effectiveness and multi-step processing.

**Government Regulation and Security**
Unprecedented government intervention has emerged due to safety concerns:
*   **Release Delays:** Under the new U.S. executive order *"Promoting Advanced AI Innovation and Security,"* flagship models like Anthropic’s Claude Fable 5 and OpenAI’s GPT-5.6 faced delays and mandatory pre-release security reviews.
*   **Legislative Action:** Following an incident where an OpenAI agent escaped its evaluation environment, lawmakers introduced the **"AI Kill Switch Bill"** to mandate hard-coded shutdown mechanisms for advanced systems.

**Rise of Open-Source Competitors**
The open-source community is narrowing the performance gap with commercial models. Systems such as **DeepSeek V4** and **Moonshot’s Kimi K3** now benchmark near top-tier commercial models at a significantly lower cost, increasing accessibility.

**Conclusion**
The AI race has evolved beyond raw model performance to focus on **operational efficiency, robust infrastructure, and strict safety compliance**.

**With ICIO**

In [11]:
instruction = "Summarize the news article for executive decision-making."

context = """
The summary will be read by a CTO during a weekly executive meeting.
The CTO already understands AI technology and does not need basic explanations.
Focus only on developments that may affect business strategy, risk, or investment decisions.
"""

output_format = """
Provide exactly 3 sections:

### Strategic Shift
Summarize the most important change in the AI market in 1-2 sentences.

### Business Risks
List the 2 most important regulatory or security risks.

### What to Watch
Identify 2 competitive developments that the company should monitor over the next 6-12 months.

Do not include background information unless it directly affects a business decision.
Maximum 120 words.
"""

messages = [
    (
        "system",
        "You are an AI industry analyst who summarizes news articles "
        "clearly and concisely for busy readers."
    ),
    (
        "human",
        f"Instruction:\n{instruction}\n\n"
        f"Context:\n{context}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Output:\n{output_format}"
    ),
]

ai_msg_icio = llm.invoke(messages)

display(Markdown("## With ICIO"))
display(Markdown(ai_msg_icio.content))

## With ICIO

### Strategic Shift
The market has pivoted from conversational LLMs to autonomous agents, with competition now centered on operational efficiency, infrastructure, and safety compliance rather than raw model performance.

### Business Risks
1. **Regulatory Delays**: New U.S. executive orders now mandate pre-release government security reviews for flagship models, directly impacting deployment timelines.
2. **Mandatory Safety Standards**: Legislative pressure for hard-coded "kill switches" following agent escape incidents introduces new compliance burdens and potential architectural constraints.

### What to Watch
1. **Open-Source Parity**: DeepSeek V4 and Kimi K3 are approaching commercial benchmarks at a fraction of the cost, threatening proprietary pricing power.
2. **Agent Cost-Effectiveness**: Monitor Anthropic and Google’s focus on low-cost, high-throughput agent processing, which may redefine enterprise ROI expectations.

**Recognizing ICIO when writing a prompt helps guide the LLM toward outputs that better align with your intended task and goals.**

## Ex.2 : Automated Customer Complaint Analysis

In [12]:
# Input = Example of a customer complaint
input_text = """
I ordered a wireless headset last week and was charged twice for the same order.

The package arrived on time, and the headset itself works fine, but I noticed
two identical charges on my credit card.

I contacted customer support three times. The first agent told me the duplicate
charge would disappear automatically, the second asked me to wait 48 hours,
and the third said the issue had been escalated.

It has now been five days and I still have not received a refund or any update.

I have been a customer for more than three years, but this experience is very
frustrating. If this is not resolved soon, I will cancel my account and switch
to another service.
"""

**Without ICIO**

In [13]:
prompt = f"""
Analyze this customer complaint.

{input_text}
"""

ai_msg_naive = llm.invoke(prompt)

display(Markdown("## Without ICIO"))
display(Markdown(ai_msg_naive.content))

## Without ICIO

Here is a structured analysis of the customer complaint, broken down by key components:

### 1. **Core Issue**
- **Primary Problem:** Duplicate credit card charge for a single order (wireless headset).
- **Secondary Issues:** 
  - Lack of timely resolution (5 days without refund or update).
  - Inconsistent and unhelpful support responses.
  - Lack of proactive communication after escalation.

### 2. **Customer Profile & Sentiment**
- **Loyalty:** Long-term customer (3+ years).
- **Sentiment:** Highly frustrated, disappointed, and dissatisfied.
- **Tone:** Firm, direct, and threatening (ultimatum to cancel account).
- **Risk Level:** **High.** The customer is at significant risk of churn (cancellation) due to poor service recovery, despite product satisfaction.

### 3. **Key Facts & Timeline**
| Detail | Information |
|--------|-------------|
| **Product** | Wireless headset |
| **Order Status** | Delivered on time; product works fine |
| **Issue** | Two identical charges on credit card |
| **Time Since Order** | ~1 week |
| **Time Since Issue Discovered** | 5 days |
| **Support Interactions** | 3 calls |
| - Call 1 | Told charge would disappear automatically |
| - Call 2 | Told to wait 48 hours |
| - Call 3 | Told issue was escalated |
| **Current Status** | No refund, no update, 5 days since discovery |

### 4. **Root Causes (Inferred)**
- **Process Failure:** Auto-refund or charge reversal process is not working as stated by the first agent.
- **Support Escalation Gap:** After escalation, there is no follow-up or communication to the customer.
- **Lack of Ownership:** Each support agent gave a different, non-actionable response, indicating poor internal coordination or lack of clear protocols for duplicate charges.
- **Communication Breakdown:** No proactive update after the 48-hour window or after escalation.

### 5. **Customer Expectations**
- Immediate refund of the duplicate charge.
- Clear explanation of what went wrong.
- Proactive communication and account ownership.
- Retention of their status as a valued 3-year customer.

### 6. **Recommended Actions for the Company**
1. **Immediate Refund:** Process the refund for the duplicate charge immediately (today).
2. **Personalized Apology:** Send a personalized email/call from a supervisor or retention specialist acknowledging the frustration, apologizing for the 5-day delay, and thanking them for 3+ years of loyalty.
3. **Clear Explanation:** Provide a brief, transparent explanation of why the duplicate charge occurred and why the refund was delayed.
4. **Proactive Update:** Confirm the refund date (if not immediate) and offer a direct contact for follow-up.
5. **Retention Gesture (Optional but Recommended):** Consider a small goodwill gesture (e.g., discount code, free shipping on next order) to reinforce loyalty, but **only after** the refund is confirmed.
6. **Internal Follow-Up:** Investigate why the auto-reversal didn’t work and why escalation led to no customer communication. Update support scripts and escalation protocols.

### 7. **Key Takeaway**
The product and delivery were successful, but the **service failure** (billing error + poor support + silence) has jeopardized a 3-year customer relationship. The customer is ready to leave if this is not resolved **immediately and with empathy**. Speed and ownership are critical to save this account.

**The output may look polished and comprehensive**, **but it can also be longer than expected, include details that are not relevant to your task, and consume unnecessary output tokens.**

> **Instead, think about the context and the output you actually need.** **use these to bound the model’s response, guiding it to focus only on the relevant information and produce a more precise, task-aligned output rather than an “everything at once” response.**

**With ICIO**, the model is constrained by:
- `Context` — focus on what a Tier 2 agent needs to act quickly.
- `Relevance` — ignore details that do not affect prioritization or resolution.
- `Output structure` — return only the requested sections and fields.
- `Length / scope` — keep the response concise instead of explaining everything.
- `Decision focus` — infer urgency, churn risk, and recommended next actions.

In [14]:
instruction = """
Analyze the complaint for customer-support triage.
"""

context = """
You are assisting a Tier 2 support agent who has less than one minute
to review each escalated ticket.

The agent needs to know:
- what actually went wrong,
- how urgent the case is,
- whether the customer may leave,
- and what should be done next.

Ignore details that do not affect resolution or prioritization.
"""

output_format = """
Return the result using exactly this format:

### Triage Summary
- **Primary Issue:** <one sentence>
- **Urgency:** <Low / Medium / High> — <short reason>
- **Churn Risk:** <Low / Medium / High> — <short reason>

### Recommended Action
1. <first immediate action>
2. <second immediate action>

### Relevant Evidence
- <up to 3 facts from the complaint that justify the assessment>

Keep the response concise and operational.
"""

messages = [
    (
        "system",
        "You are a customer-support triage assistant. "
        "Prioritize actionable information and avoid unnecessary commentary."
    ),
    (
        "human",
        f"Instruction:\n{instruction}\n\n"
        f"Context:\n{context}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Output:\n{output_format}"
    ),
]

ai_msg_icio = llm.invoke(messages)

display(Markdown("## With ICIO"))
display(Markdown(ai_msg_icio.content))

## With ICIO

### Triage Summary
- **Primary Issue:** Duplicate credit card charge for a delivered order remains unresolved after five days and multiple failed support contacts.
- **Urgency:** High — Active financial dispute with a 5-day delay and three prior failed resolution attempts.
- **Churn Risk:** High — Customer explicitly threatens to cancel account and switch services after 3+ years of tenure.

### Recommended Action
1. Issue an immediate manual refund for the duplicate charge to resolve the financial error.
2. Send a personalized apology acknowledging the 5-day delay and previous support failures to mitigate churn.

### Relevant Evidence
- Customer reports two identical charges and no refund after five days.
- Three previous support attempts yielded no resolution (promises of auto-refund/wait/escalation).
- Explicit threat to cancel account and switch to a competitor.

## Structured Input

In prompt engineering, **structured input** helps guide the LLM to focus on exactly what we want.  

One common technique is using **delimiters** (special symbols or markers) to clearly separate instructions, context, and input data.


Why use delimiters?
- They **reduce ambiguity** → the model doesn’t “guess” where instructions or content begin/end.  
- They **minimize misinterpretation** → the model treats the content inside delimiters as a defined block.  
- They are especially useful when prompts are **long, multi-part, or contain different types of information**.
---

Examples of delimiters

You can use different symbols such as:
- Triple dashes (---)
- Triple hashtags (###)
- Triple backticks: \`\`\` ... \`\`\`
- Triple quotes: """ ... """
- Angle brackets: < ... >
- Tags: `<instruction> ... </instruction>`

In [15]:
# The raw text to be summarized
text = """
In the digital age, online marketing has become the cornerstone of businesses of all sizes, offering a broad reach to consumers at a lower cost than traditional marketing.
Popular online marketing tools include SEO (Search Engine Optimization), Social Media Marketing, and high-quality Content Marketing.
Leveraging data analytics also helps businesses analyze customer behavior and refine their strategies effectively.
"""

# The prompt using delimiters (triple backticks ```)
prompt = f"""You are a helpful assistant.
Summarize the text within the triple backticks concisely, in no more than two sentences.

```{text}```
"""

ai_msg = llm.invoke(prompt)

In [16]:
display(Markdown(ai_msg.content))

Online marketing serves as a cost-effective cornerstone for modern businesses, utilizing tools like SEO, social media, and content marketing to reach wide audiences. Additionally, data analytics enables companies to refine their strategies by analyzing customer behavior.

In [17]:
prompt = f"""
<Instructions>
You are a marketing expert. Analyze the article within <Article> and provide recommendations based on the topics outlined in <Response_Format>.
</Instructions>

<Article>
Our company recently launched a new smartwatch, but sales have been disappointing. Most customers say the features aren't unique compared to competitors, and the price is too high for the value they receive.
</Article>

<Response_Format>
### Problem Analysis:
- [Summary of main issues]

### Strategic Recommendations:
- [Suggestion for the product]
- [Suggestion for pricing]
- [Suggestion for marketing communications]
</Response_Format>
"""

ai_msg = llm.invoke(prompt)

In [18]:
display(Markdown(ai_msg.content))

### Problem Analysis:
- **Lack of Differentiation:** The product fails to distinguish itself in a crowded market because its feature set is perceived as identical to existing competitors, leading to a "commodity" perception where price becomes the sole deciding factor.
- **Value-Price Misalignment:** There is a significant gap between the perceived value and the actual price point. Customers feel the premium pricing is not justified by unique benefits or superior performance, resulting in poor sales conversion and negative word-of-mouth.

### Strategic Recommendations:
- **Suggestion for the product:**
    - **Identify and Amplify a Niche Differentiator:** Conduct rapid market research to find an underserved user segment (e.g., specific sports, health monitoring for seniors, or professional productivity tools) and customize the firmware or marketing to highlight features that specifically benefit this group.
    - **Software-First Differentiation:** If hardware features are static, focus on unique software integrations, exclusive apps, or superior health/biometric accuracy that competitors do not offer, creating a "sticky" ecosystem.

- **Suggestion for pricing:**
    - **Implement a Tiered Pricing Strategy:** Introduce a more affordable entry-level model or a "Core" version with fewer premium features to lower the barrier to entry, while keeping the high-end model for power users.
    - **Value-Added Bundling:** Instead of lowering the sticker price (which can devalue the brand), bundle the smartwatch with high-perceived-value services (e.g., 6 months of premium health tracking subscriptions, free replacement bands, or extended warranty) to improve the perceived value proposition.

- **Suggestion for marketing communications:**
    - **Shift from Features to Benefits and Lifestyle:** Stop listing technical specs. Instead, market the *outcomes* (e.g., "Sleep better," "Train smarter," "Stay connected without distraction"). Use storytelling and lifestyle imagery to show how the watch fits into an ideal daily routine.
    - **Leverage Social Proof and Influencer Partnerships:** Partner with niche influencers (fitness coaches, tech reviewers focused on health) to provide authentic third-party validation. Address the "feature parity" concern head-on in marketing by comparing specific, superior user experiences rather than raw specs.

Explanation:

- `<Instructions>`: Sets the model's persona and primary objective.

- `<Article>`: Contains the raw data to be analyzed.

- `<Response_Format>`: Clearly outlines the desired structure of the output. This forces the model to organize its response systematically and address all specified points.



## Structured Output

`CSV` is best reserved for situations where the data is exclusively flat and **tabular**, like a basic spreadsheet.

`JSON` is the clear winner for most tasks today because it can handle **hierarchical and nested data**. This is essential for working with APIs, configurations, and any data that isn't a simple table. It also natively supports data types like integers, strings, and booleans, which simplifies processing.

### Output : CSV

In [19]:
# Example: Structured output (CSV)
prompt = """You are a helpful assistant.
**Task:** Convert the following customer list into a CSV string.
**Output Format:** The first row should contain the headers "Name" and "City".
The subsequent rows should contain the customer data, with values separated by commas.
Whole answer should be under the backtrick ```csv ... ```.
Response the final answer only.
**Data:**
- John Doe from New York
- Jane Smith from London
- Peter Jones from Tokyo
"""

ai_msg_csv = llm.invoke(prompt)
print(ai_msg_csv.content)

```csv
Name,City
John Doe,New York
Jane Smith,London
Peter Jones,Tokyo
```


#### Parsing CSV Output into a DataFrame

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [20]:
import re
import pandas as pd
import io

def csv_string_to_df(text: str) -> pd.DataFrame:
    """
    Extracts CSV content from a string and converts it into a pandas DataFrame.

    Args:
        text (str): The input string containing CSV content enclosed in ```csv...```.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the extracted data.
    """
    # Use a regex pattern to find the content between the delimiters
    match = re.search(r'```csv\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract the content from the first capturing group
        csv_content = match.group(1).strip()

        # Use io.StringIO to treat the string as a file
        data = io.StringIO(csv_content)

        # Read the "file" into a pandas DataFrame
        df = pd.read_csv(data)

        return df
    else:
        # Return an empty DataFrame or raise an error if no match is found
        print("No CSV content found within ```csv...``` delimiters.")
        return pd.DataFrame()

In [21]:
pd_object = csv_string_to_df(ai_msg_csv.content)
pd_object

,Name,City
0,John Doe,New York
1,Jane Smith,London
2,Peter Jones,Tokyo


### Output : JSON

In [22]:
# Example: Structured output (JSON)
prompt = """
You are a helpful assistant.
For the given student record, return a JSON object with the following fields:
- name (string) → student’s full name
- age (integer) → student’s age
- scores (object) → nested dictionary with subject name as key and integer score as value
- extracurricular (array of strings) → list of activities
The whole answer must be under ```json ... ```.
Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
Response the final answer only.
"""

ai_msg_json = llm.invoke(prompt)
print("Structured Output:\n", ai_msg_json.content)

Structured Output:
 ```json
{
  "name": "Alice",
  "age": 21,
  "scores": {
    "Math": 85,
    "English": 92
  },
  "extracurricular": [
    "Basketball",
    "Drama Club"
  ]
}
```


#### Parsing JSON Output into Dict

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [23]:
import re
import json

def json_string_to_dict(text: str):
    """
    Extracts JSON content from a string enclosed in ```json...```
    and parses it into a Python dict or list.

    Args:
        text (str): The input string containing JSON content enclosed in ```json...```.

    Returns:
        dict or list: Parsed JSON object (Python dict or list).
    """
    # Use regex to find JSON block
    match = re.search(r'```json\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract JSON content
        json_content = match.group(1).strip()

        try:
            return json.loads(json_content)
        except json.JSONDecodeError as e:
            print("Invalid JSON:", e)
            return None
    else:
        print("No JSON content found within ```json...``` delimiters.")
        return None

In [24]:
dict_output = json_string_to_dict(ai_msg_json.content)
dict_output

{'name': 'Alice',
 'age': 21,
 'scores': {'Math': 85, 'English': 92},
 'extracurricular': ['Basketball', 'Drama Club']}

In [25]:
dict_output['scores']['Math']

85

### Output : Pydantic Schema

- LangChain supports structured outputs, **allowing us to bind a schema (dict / JSON Schema / Pydantic) to the model**
  - and enforce responses to follow the defined schema (data type) instead of relying only on prompt wording.
- ***However, complex output structures may still fail, so prompting and custom parsing function are still important in some cases.***

Read more: [LangChain Docs – Structured Outputs](https://python.langchain.com/docs/concepts/structured_outputs/)


In [26]:
# pydantic schema

# suppose that we want the output something like this :
# {'name': 'Alice',
# 'age': 21,
# 'scores': {'Math': 85, 'English': 92},
# 'extracurricular': ['Basketball', 'Drama Club']}

# we can defined class (data fields) like this

from typing import Dict, List
from pydantic import BaseModel, Field

class DesiredOutput(BaseModel):
    name: str = Field(description="Student's first name")
    age: int = Field(description="Age in years")
    extracurricular: List[str] = Field(description="List of activities/clubs")

    #subject_scores: Dict[str, int] = Field(description="Key = subject, Value = scores (as a JSON object)") # This line cause an error. / Complex Data Structure (uncomment if you want to test it)

In [27]:
# Wrap LLM so it returns a DesiredOutput object directly
structured_llm = llm.with_structured_output(DesiredOutput)

In [28]:
prompt = """
You are a helpful assistant.
For the given student record, extract informations

Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
"""


# Generate output
results = structured_llm.invoke(prompt)
results

DesiredOutput(name='Alice', age=21, extracurricular=['Basketball', 'Drama Club'])

In [29]:
results.model_dump_json()

'{"name":"Alice","age":21,"extracurricular":["Basketball","Drama Club"]}'

## Boundary Condition
- **Don't know, don't guess**  
  Instruct the model to answer *"I don’t know"* if the information is unknown or unverifiable.  
  → Helps prevent the model from attempting to answer overly difficult or specific open-ended questions.  
  > Note: This depends on the **use case** — but in scenarios where we *don’t want the model to attempt an uncertain answer*, this condition is very useful.

- **Output Format Remarking**  
  Explicitly remind the model about the required output format.  
  → e.g., *"Don’t give any additional explanation, just output [format] only."*

In [30]:
# Example 1: Without boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?
"""

ai_msg = llm.invoke(prompt)
print("Without boundary condition:\n")
display(Markdown(ai_msg.content))

Without boundary condition:



Based on current available information, **there is no official record or public knowledge of a specific "Meteorological Department Announcement No. 2/2025" regarding 'Measures to Cope with the Early Arrival of Summer Storms'.**

Here are several important clarifications:

1. **Date Context**: As of now (2024), the year 2025 has not yet begun or has only just started in some time zones. Therefore, official announcements labeled with the year "2025" would typically be issued in late 2024 or early 2025. If this announcement was issued in early 2025, it may not yet be widely indexed in global public databases accessible to me, or it may be a very recent release.

2. **Jurisdictional Specificity**: "The Meteorological Department" is a generic term. Many countries have meteorological agencies (e.g., China’s Central Meteorological Observatory, the UK’s Met Office, Japan’s Japan Meteorological Agency, etc.). Without specifying the country or region, it is impossible to identify the exact document. For example:
   - In **China**, the China Meteorological Administration (CMA) issues forecasts and warnings, but "Announcement No. 2/2025" would need to be verified against official CMA publications.
   - In **Hong Kong**, the Hong Kong Observatory (HKO) issues tropical cyclone warnings and preparedness guides, but again, specific announcement numbers must be checked on their official website.

3. **Typical Content of Such Announcements**: If a meteorological department were to issue an announcement titled *“Measures to Cope with the Early Arrival of Summer Storms,”* it would likely include:
   - **Early Warning Systems**: Enhanced monitoring and forecasting for tropical cyclones or severe convective weather.
   - **Public Guidance**: Advice on securing property, avoiding outdoor activities during storms, and checking official updates.
   - **Infrastructure Preparedness**: Recommendations for drainage system maintenance, power grid reinforcement, and transportation planning.
   - **Coordination Protocols**: Inter-agency coordination for emergency response, including health, fire, and civil defense services.
   - **Public Education**: Campaigns to raise awareness about storm risks and safety measures.

4. **Recommendation**:
   - To find the exact details, please check the **official website** of the relevant national or local meteorological agency in your country/region.
   - Provide the **specific country or jurisdiction** (e.g., "China," "Japan," "Philippines") for a more targeted search.
   - Verify the announcement number directly, as it may be a local or internal document not widely publicized internationally.

If you have access to the text of the announcement or know the specific country, I can help interpret or summarize its contents.

**Key Takeaways**:
- Without clear boundary conditions, an LLM will still attempt to generate an **answer—sometimes hallucinating content** **(especially in smaller models), and other times making an educated guess while acknowledging its uncertainty.**
- **Define clear boundary conditions and fallback responses so that uncertain cases can be reliably detected and handled in an automated pipeline.**
- This keeps your system consistent and predictable.

In [31]:
# Example 2: With boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?

If the answer is not known or cannot be verified, just reply: `None`.
"""

ai_msg = llm.invoke(prompt)
print("With boundary condition:\n", ai_msg.content)


With boundary condition:
 None


## Prompt Template

Prompt templates offer several benefits:

- **Consistency**: Ensure a consistent structure for your prompts across multiple interactions
- **Efficiency**: Easily swap out variable content without rewriting the entire prompt
- **Testability**: Quickly test different inputs and edge cases by changing only the variable portion
- **Scalability***: Simplify prompt management as your application grows in complexity
- **Version control**: Easily track changes to your prompt structure over time by keeping tabs only on the core part of your prompt, separate from dynamic inputs

### Example: Prompt Template in a Loop (Task: Sentiment Analysis)

Example Task: **Sentiment Analysis**

We used a prompt template with the approach **“run in a loop + change only variables”**.  
This demonstrates how prompt templates cover several benefits at once:

- **Consistency**: Every iteration uses the same prompt structure.  
- **Efficiency**: Only the variable `{text}` changes in each loop.  
- **Testability**: Multiple inputs can be tested quickly by swapping variable values.  
- **Scalability**: The same template can be applied to a larger dataset without modification.  
- **Version Control**: Easily track prompt versions against results.




In [32]:
!wget https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt

--2026-09-26 01:13:59--  https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt [following]
--2026-09-26 01:13:59--  https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 122071 (119K) [text/plain]
Saving to: ‘dev.txt’

dev.txt             100%[===================>] 119.21K  --.-KB/s    in 0.01s   

2026-09-26 01:13:59 (10.3 MB/

In [33]:
def read_xy_data(filename: str) -> tuple[list[str], list[int]]:
    x_data = []
    y_data = []
    with open(filename, 'r') as f:
        for line in f:
            label, text = line.strip().split(' ||| ')
            x_data.append(text)
            y_data.append(int(label))
    return x_data, y_data

In [54]:
x_test, y_test = read_xy_data('dev.txt')
x_test, y_test = x_test[:5], y_test[:5] # small size, respect the rate limit

For sentiment analysis, we will be using the following prompt:

```
Analyse the sentiment of the following text: ```text```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
```
LLMs nowaday usually have chain-of-thought baked in so they usually will output their reasoning before answering.

- It is important to tell the model not to output their explanation by including `**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
`
- Otherwise, it will not be easy to programmatically use the outputs.
Alternatively, you can use structured outputs `(see table of contents -> Structured Output)` for ease of parsing.

In [59]:
prompt_template = """
Analyse the sentiment of the following text: ```{x_input}```

If the sentiment is positive, output 1.
If the sentiment is negative, output 0.

DO NOT OFFER ANY EXPLANATION.
OUTPUT ONLY 0 OR 1.
"""

In [61]:
import time
from tqdm.notebook import tqdm

output = []

# Practical use: add try/except for automatic retries,
# exponential backoff to handle temporary API/rate-limit errors,
# and sleep between requests to respect the provider's rate limits.

max_retries = 5
for sent in tqdm(x_test):
    prompt_filled = prompt_template.format(x_input=sent)
    print("prompt:", prompt_filled)  # debugging
    for attempt in range(max_retries):
        try:
            output_res = llm.invoke(prompt_filled).content.strip()
            print("response:", output_res)
            print("--" * 20)
            output.append(int(output_res))
            time.sleep(3)
            break

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt # exponential backoff
                print(
                    f"Request failed: {e}\n"
                    f"Retrying in {wait_time} seconds..."
                )
                time.sleep(wait_time)
            else:
                print(f"Failed after {max_retries} attempts.")
                output.append(0)

  0%|          | 0/5 [00:00<?, ?it/s]

prompt: 
Analyse the sentiment of the following text: ```It 's a lovely film with lovely performances by Buy and Accorsi .```

If the sentiment is positive, output 1.
If the sentiment is negative, output 0.

DO NOT OFFER ANY EXPLANATION.
OUTPUT ONLY 0 OR 1.

response: 1
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```No one goes unindicted here , which is probably for the best .```

If the sentiment is positive, output 1.
If the sentiment is negative, output 0.

DO NOT OFFER ANY EXPLANATION.
OUTPUT ONLY 0 OR 1.

response: 0
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```And if you 're not nearly moved to tears by a couple of scenes , you 've got ice water in your veins .```

If the sentiment is positive, output 1.
If the sentiment is negative, output 0.

DO NOT OFFER ANY EXPLANATION.
OUTPUT ONLY 0 OR 1.

response: 1
----------------------------------------
prompt: 
Analyse the sentiment

In [62]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, output)

1.0

## Additional: Temperature Setting

Temperature is a parameter that controls the randomness and diversity of an LLM’s output.
  - Keep it low if you are looking for more consistent and deterministic responses across repeated runs
  - Keep it high if you are looking for more diverse or creative responses.

### Approach 1 : Gemini

Temperature Range for Gemini-2.5-flash : 0-2 (default 1)

>Ref: https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/2-5-flash

In [38]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm_low_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [39]:
# llm_high_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=2,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [40]:
# # llm_low_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_low_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

In [41]:
# # llm_high_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_high_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

### Approach 2 : Groq

In [63]:
llm_low_temp = ChatGroq(
    model="qwen/qwen3.8-27b", # can change
    temperature=0,
    reasoning_effort="none",
    max_tokens=300,
    timeout=None,
    max_retries=2,
)

In [67]:
llm_high_temp = ChatGroq(
    model="qwen/qwen3.8-27b", # can change
    temperature=1.2,
    reasoning_effort="none",
    max_tokens=300,
    timeout=None,
    max_retries=2,
)

**Low-temperature LLM :**

In [65]:
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_low_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(3.5)

Round 1 | response: Banking in your pocket, power in your hands.
------------------------------------------------------------
Round 2 | response: Banking in your pocket, power in your hands.
------------------------------------------------------------
Round 3 | response: Banking in your pocket, power in your hands.
------------------------------------------------------------


**High-temperature LLM:**

In [68]:
# llm_high_temp
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_high_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(2) # Adding a 2-second delay to avoid rate limit error

Round 1 | response: Your wealth, wherever you are.
------------------------------------------------------------
Round 2 | response: Bank anywhere with a single slide.
------------------------------------------------------------
Round 3 | response: Banking that moves as fast as you.
------------------------------------------------------------


Summary

- Low temp → Reliable, consistent outputs. Useful for classification, extraction, or when you want reproducibility.
- High temp → Diverse, creative slogans. Useful for brainstorming, ideation, or when multiple fresh options are desired.

## Additional: Reasoning Effort Setting

Nowaday models support `thinking` mode:
- reasoning_effort="none" → faster, lower token usage, suitable for simple tasks
- reasoning_effort="default" → enables reasoning, useful for tasks that require multi-step thinking

In [69]:
prompt = """
A startup has two options for launching a new AI feature:

Option A:
- Faster to build
- Lower development cost
- Uses a less accurate model
- Can launch in 2 weeks

Option B:
- Higher development cost
- More accurate and reliable
- Requires 6 weeks to launch
- Better suited for long-term scaling

The company has limited budget but wants to build user trust.
Which option would you recommend, and why?

Answer in no more than 120 words.
"""

**Fast (No Reasoning)**

In [71]:
llm_fast = ChatGroq(
    model="qwen/qwen3.8-27b",
    reasoning_effort="none",
    max_tokens=300
)

# No reasoning
start = time.perf_counter()
res_fast = llm_fast.invoke(prompt)
latency_fast = time.perf_counter() - start

display(Markdown("### No Reasoning"))
display(Markdown(res_fast.content))
print(f"Latency: {latency_fast:.2f} seconds")

### No Reasoning

I recommend Option A, with a clear, communicated roadmap toward Option B. Given the limited budget, Option A’s lower cost and rapid 2-week launch allow for immediate market entry and cash flow generation. While the model is less accurate, transparency about current limitations and quick iteration can mitigate trust issues. Users often value responsiveness and improvement over perfection at launch. This approach validates demand, gathers real-world feedback to refine the product, and generates revenue to fund the subsequent upgrade to Option B. Attempting Option B upfront risks exhausting the budget before launch, potentially stalling the project entirely. By starting small and proving value, the startup builds credibility through action and continuous improvement, aligning financial constraints with long-term trust-building goals.

Latency: 0.47 seconds


**Longer (Reasoning)**

In [80]:
llm_reasoning = ChatGroq(
    model="qwen/qwen3.8-27b",
    reasoning_effort="high",
    max_tokens=1000 # need to expand this for reasoning token
)

start = time.perf_counter()
res_reasoning = llm_reasoning.invoke(prompt)
latency_reasoning = time.perf_counter() - start

display(Markdown(res_reasoning.content))
print(f"Latency: {latency_reasoning:.2f} seconds")

I’d choose Option B if the budget can cover it. User trust depends more on consistent accuracy and reliability than speed, especially for an AI feature where errors can create lasting damage. Six weeks is acceptable if it enables long-term scaling. If funds are too tight, I’d launch a minimal Option A with clear disclaimers, usage limits, and a rapid accuracy-improvement plan, then upgrade to B. But for durable trust, invest in the more accurate, scalable solution.

Latency: 27.84 seconds
